# Exercise 4.2: Recreate a published plot

From *Programming in High Energy Particle Physics*, Chapter 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch04/ex04_2_solution.ipynb)

Exercise 4.2 Recreate a published plot Using Matplotlib and mplhep, recreate one of your favorite plots from a recent CMS or ATLAS paper. Pay attention to fonts, label positioning, and color choices.

<details><summary>Hint</summary>

The numerical content of most published plots is available on HEPData ( hepdata.net ) as tables that you can download and plot with hep.histplot . Start from hep.style.use("CMS") or hep.style.use("ATLAS") and compare side by side with the original.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib mplhep requests")

### Recreate a published CMS Higgs transverse-momentum spectrum

CMS, *Measurement of the Production Cross Section of a Higgs Boson with Large Transverse Momentum in Its Decays to a Pair of $\tau$ Leptons in Proton-Proton Collisions at $\sqrt{s}$ = 13 TeV* ([arXiv:2403.20201](https://arxiv.org/abs/2403.20201), DOI: 10.1016/j.physletb.2024.138964). Use HEPData [record 149577](https://www.hepdata.net/record/149577), exact table `Diff XS v.s. Higgs pt`. Its observed distribution has four bins from 250 to 2000 GeV. The table headers are `p_{T}^{H} [GeV]` and `Observed [fb/GeV]`.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

TABLE_URL = "https://www.hepdata.net/record/data/149577/1643683/1/"
try:
    response = requests.get(TABLE_URL, timeout=30)
    response.raise_for_status()
    table_data = response.json()
except requests.RequestException as exc:
    print(f"HEPData unavailable ({exc}). Connect to the network and rerun this cell.")
    table_data = None

if table_data is not None:
    assert table_data["name"] == "Diff XS v.s. Higgs pt"
    rows = table_data["values"]
    edges = np.array([float(row["x"][0]["low"]) for row in rows] +
                     [float(rows[-1]["x"][0]["high"])])
    values = np.array([float(row["y"][0]["value"]) for row in rows])
    errors = np.array([float(row["y"][0]["errors"][0]["symerror"]) for row in rows])
    x_header, y_header = (header["name"] for header in table_data["headers"][:2])
    hep.style.use("CMS")
    fig, ax = plt.subplots(figsize=(8, 5))
    hep.histplot(values, bins=edges, yerr=errors, histtype="errorbar", ax=ax,
                 color="black", label="Observed")
    ax.set_xlabel(x_header)
    ax.set_ylabel(y_header)
    ax.set_xlim(edges[0], edges[-1])
    ax.legend(frameon=False)
    hep.cms.label(data=True, ax=ax)
    ax.text(0.98, 0.95, "H → ττ · √s = 13 TeV", ha="right", va="top",
            transform=ax.transAxes, fontsize=10)
    plt.show()
    print("HEPData table:", table_data["name"])
    print("Bins:", len(values), "range:", edges[0], edges[-1], "GeV")


In [ ]:
if table_data is not None:
    assert len(edges) == len(values) + 1 and np.all(np.diff(edges) > 0)
else:
    print("Table not loaded; rerun with network access.")

### Interpretation

The four observed points and their total uncertainties come directly from the HEPData table. They cover only the large-$p_T(H)$ region (250–2000 GeV) measured in this $H\to\tau\tau$ analysis. The plot uses the table's exact axis headers and units.